# 붓꽃(iris)의 품종 분류

In [1]:
# '꽃잎과 꽃받침의 크기' 를 기반으로 붓꽃의 품종 분류

# 데이터셋 : iris.csv   

# 컬럼
# SepalLength  (꽃받침의 길이)
# SepalWidth   (꽃받침의 폭)
# PetalLength  (꽃잎의 길이)
# PetalWidth   (꽃임이 너비)

# Name     품종명 (Species)⭐
#     "Iris-setosa", "Iris-versicolor", "Iris-virginica"  세가지 품종

# 출처 : https://www.kaggle.com/uciml/iris

In [2]:
# CSV 파일에는 약 150개의 데이터가 있는데,  
# 100개는 학습(train)을 위해 사용,   50개는 테스트(test)를 위해 사용

In [3]:
""" 
[수행 단계]

① 필요한 import 수행

② 데이터 읽어오기
   CSV -> DataFrame (변수명 df)
   기초 통계랑 확인
   
③ 입력데이터와 타겟데이터 분리
   입력데이터 →  변수명 data
   타겟데이터 →  변수명 target

④ train, test 세트 분리. 
   - 8:2로 분류 
   - 클래스별로 균등하게 분류되게 하고 

⑤ 전처리 (표준화)
   스케일러 객체 변수 → 변수명 scaler
  
⑥ 최적의 하이퍼 파라미터 찾기
   SVC 의 최적 하이퍼 파라미터 찾기 수행.  GridSearchCV 사용

   param_grid = {'C': [0.1, 1, 10, 100],               
              'gamma': [0.001, 0.1, 1, 10, 100], 
              'kernel': ['linear', 'rbf']}

   최적의 모델 저장 -> 변수명 clf

⑦ test 점수 확인
    
⑧ 다른 평가지표들 확인

⑨ 예측 동작 확인

⑩ 모델 & 스케일러 저장하기
   모델 -> 파일명 iris_model.pkl
   스케일러 -> 파일명 iris_scaler.pkl

   (위 저장 정보는 웹 애플리케이션에서 사용될것임)

⑪ 저장된 모델 & 스케일러 불러오기

⑫ 예측 함수 만들어 보고 동작 시키기

    # 예측 함수 작성
    # 입력값: 웹에서 사용자가 입력한 값
    # 출력값: 분류 문자열 (ex: Iris-setosa, Iris-versicolor, Iris-virginica)
    def predict_iris(sepal_length, sepal_width, petal_length, petal_width) -> str:


★ 각 수행 단계별로 적절한 제목으로 작성
★ 각 수행 단계마다 이 단계가 무엇을 하는 단계이고, 사용하는 파라미터와
   (필요한 경우) 어떠한 입력으로 어떠한 결과가 나오는지 확인하고 설명을 남기기
★ 각 단계마다 내가 무엇을 확인했는지 코드와 함께 설명 남기기
"""
None

# ① 필요한 import 수행

In [45]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn import svm, metrics
import joblib   # 학습 모델 저장  pip install joblib
import glob, os, re, json

import pandas as pd

# ② 데이터 읽어오기
   CSV -> DataFrame (변수명 df)
   기초 통계랑 확인

In [46]:
df = pd.read_csv(os.path.join('.', 'iris.csv'))
df

,SepalLength,SepalWidth,PetalLength,PetalWidth,Name
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,Iris-virginica
146,6.3,2.5,5.0,1.9,Iris-virginica
147,6.5,3.0,5.2,2.0,Iris-virginica
148,6.2,3.4,5.4,2.3,Iris-virginica


In [47]:
df.shape

(150, 5)

In [48]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   SepalLength  150 non-null    float64
 1   SepalWidth   150 non-null    float64
 2   PetalLength  150 non-null    float64
 3   PetalWidth   150 non-null    float64
 4   Name         150 non-null    str    
dtypes: float64(4), str(1)
memory usage: 7.9 KB


In [49]:
df.describe()

,SepalLength,SepalWidth,PetalLength,PetalWidth
count,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.054000,3.758667,1.198667
std,0.828066,0.433594,1.764420,0.763161
min,4.300000,2.000000,1.000000,0.100000
25%,5.100000,2.800000,1.600000,0.300000
50%,5.800000,3.000000,4.350000,1.300000
75%,6.400000,3.300000,5.100000,1.800000
max,7.900000,4.400000,6.900000,2.500000


In [50]:
df.Name.unique()

<ArrowStringArray>
['Iris-setosa', 'Iris-versicolor', 'Iris-virginica']
Length: 3, dtype: str

# ③ 입력데이터와 타겟데이터 분리
   입력데이터 →  변수명 data
   타겟데이터 →  변수명 target

In [51]:
data = df[['SepalLength', 'SepalWidth', 'PetalLength', 'PetalWidth']]

print(data.shape)
data[:5]

(150, 4)


,SepalLength,SepalWidth,PetalLength,PetalWidth
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


In [52]:
target = df['Name']

print(target.shape)
target

(150,)


0         Iris-setosa
1         Iris-setosa
2         Iris-setosa
3         Iris-setosa
4         Iris-setosa
            ...      
145    Iris-virginica
146    Iris-virginica
147    Iris-virginica
148    Iris-virginica
149    Iris-virginica
Name: Name, Length: 150, dtype: str

# ④ train, test 세트 분리. 
   - 8:2로 분류 
   - 클래스별로 균등하게 분류되게 하고 

In [54]:
X_train, X_test, y_train, y_test = \
    train_test_split(data, target, test_size=0.2, stratify=target, random_state=42)

X_train.shape, X_test.shape, y_train.shape, y_test.shape   

((120, 4), (30, 4), (120,), (30,))

In [55]:
y_train.value_counts()

Name
Iris-setosa        40
Iris-virginica     40
Iris-versicolor    40
Name: count, dtype: int64

In [56]:
y_test.value_counts()

Name
Iris-setosa        10
Iris-virginica     10
Iris-versicolor    10
Name: count, dtype: int64

# ⑤ 전처리 (표준화)
   스케일러 객체 변수 → 변수명 scaler  

In [57]:
scaler = StandardScaler()
scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train[:5])
print()
print(X_train_scaled[:5])

     SepalLength  SepalWidth  PetalLength  PetalWidth
8            4.4         2.9          1.4         0.2
106          4.9         2.5          4.5         1.7
76           6.8         2.8          4.8         1.4
9            4.9         3.1          1.5         0.1
89           5.5         2.5          4.0         1.3

[[-1.72156775 -0.32483982 -1.34703555 -1.32016847]
 [-1.12449223 -1.22612948  0.41429037  0.65186742]
 [ 1.14439475 -0.55016223  0.58474127  0.25746024]
 [-1.12449223  0.12580502 -1.29021859 -1.45163753]
 [-0.40800161 -1.22612948  0.13020555  0.12599118]]


In [58]:
pd.DataFrame(data=X_train_scaled).describe()

,0,1,2,3
count,1.200000e+02,1.200000e+02,1.200000e+02,1.200000e+02
mean,-1.369275e-16,4.551914e-16,-9.066821e-17,5.366078e-17
std,1.004193e+00,1.004193e+00,1.004193e+00,1.004193e+00
min,-1.840983e+00,-2.352742e+00,-1.517486e+00,-1.451638e+00
25%,-8.856620e-01,-5.501622e-01,-1.233402e+00,-1.188699e+00
50%,-1.094638e-01,-9.951740e-02,2.722480e-01,1.259912e-01
75%,6.667343e-01,5.764498e-01,7.551922e-01,7.833365e-01
max,2.457961e+00,3.054996e+00,1.777898e+00,1.703620e+00


# ⑥ 최적의 하이퍼 파라미터 찾기
   SVC 의 최적 하이퍼 파라미터 찾기 수행.  GridSearchCV 사용

   param_grid = {'C': [0.1, 1, 10, 100],               
              'gamma': [0.001, 0.1, 1, 10, 100], 
              'kernel': ['linear', 'rbf']}

   최적의 모델 저장 -> 변수명 clf


In [59]:
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [0.001, 0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf'],
}

gs = GridSearchCV(
    svm.SVC(random_state=42),
    param_grid,
    n_jobs=-1,
    cv=5,
)

gs.fit(X_train_scaled, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",SVC(random_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [0.1, 1, ...], 'gamma': [0.001, 0.1, ...], 'kernel': ['linear', 'rbf']}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, default=0Controls the verbosity of infor

In [60]:
print("최적 점수 : {}".format(gs.best_score_))
print("최적 파라미터 : {}".format(gs.best_params_))

최적 점수 : 0.9833333333333332
최적 파라미터 : {'C': 1, 'gamma': 0.1, 'kernel': 'rbf'}


In [61]:
# 최적의 모델 저장
clf = gs.best_estimator_
clf

,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",0.1
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo random number generation for shuffling the data forprobability estimates. Ignored when `probability` is False.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide <scores_probabilities>`...deprecated:: 1.9 The `probability` parameter is deprecated and will be removed in 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`.",'deprecated'
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None


In [62]:
pd.DataFrame(gs.cv_results_)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_gamma,param_kernel,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.036732,0.013681,0.003044,0.001720,0.1,0.001,linear,"{'C': 0.1, 'gamma': 0.001, 'kernel': 'linear'}",0.916667,1.000000,1.000000,0.958333,1.000000,0.975000,0.033333,2
1,0.024868,0.025287,0.002687,0.001734,0.1,0.001,rbf,"{'C': 0.1, 'gamma': 0.001, 'kernel': 'rbf'}",0.875000,0.875000,0.833333,0.875000,0.875000,0.866667,0.016667,34
2,0.002700,0.001181,0.001651,0.001167,0.1,0.100,linear,"{'C': 0.1, 'gamma': 0.1, 'kernel': 'linear'}",0.916667,1.000000,1.000000,0.958333,1.000000,0.975000,0.033333,2
3,0.002754,0.000589,0.001318,0.000636,0.1,0.100,rbf,"{'C': 0.1, 'gamma': 0.1, 'kernel': 'rbf'}",0.875000,0.875000,0.833333,0.875000,0.916667,0.875000,0.026352,30
4,0.001414,0.000517,0.000626,0.000253,0.1,1.000,linear,"{'C': 0.1, 'gamma': 1, 'kernel': 'linear'}",0.916667,1.000000,1.000000,0.958333,1.000000,0.975000,0.033333,2
5,0.001557,0.000469,0.000907,0.000393,0.1,1.000,rbf,"{'C': 0.1, 'gamma': 1, 'kernel': 'rbf'}",0.916667,0.875000,0.958333,0.916667,0.958333,0.925000,0.031180,28
6,0.001785,0.000188,0.000821,0.000269,0.1,10.000,linear,"{'C': 0.1, 'gamma': 10, 'kernel': 'linear'}",0.916667,1.000000,1.000000,0.958333,1.000000,0.975000,0.033333,2
7,0.001533,0.000499,0.000712,0.000253,0.1,10.000,rbf,"{'C': 0.1, 'gamma': 10, 'kernel': 'rbf'}",0.750000,0.666667,0.791667,0.666667,0.833333,0.741667,0.066667,36
8,0.001087,0.000105,0.000476,0.000015,0.1,100.000,linear,"{'C': 0.1, 'gamma': 100, 'kernel': 'linear'}",0.916667,1.000000,1.000000,0.958333,1.000000,0.975000,0.033333,2
9,0.001554,0.000598,0.000722,0.000312,0.1,100.000,rbf,"{'C': 0.1, 'gamma': 100, 'kernel': 'rbf'}",0.375000,0.333333,0.333333,0.375000,0.375000,0.358333,0.020412,40


In [63]:
print(gs.cv_results_['mean_test_score'])

[0.975      0.86666667 0.975      0.875      0.975      0.925
 0.975      0.74166667 0.975      0.35833333 0.975      0.86666667
 0.975      0.98333333 0.975      0.95833333 0.975      0.875
 0.975      0.45833333 0.95833333 0.9        0.95833333 0.96666667
 0.95833333 0.95       0.95833333 0.875      0.95833333 0.475
 0.96666667 0.975      0.96666667 0.96666667 0.96666667 0.95
 0.96666667 0.875      0.96666667 0.475     ]


# ⑦ test 점수 확인

In [65]:
print(clf.score(X_train_scaled, y_train))
print(clf.score(X_test_scaled, y_test))


0.9833333333333333
0.9666666666666667


# ⑧ 다른 평가지표들 확인

In [66]:
predict = clf.predict(X_test_scaled)

print(predict)
print(y_test.values)

['Iris-setosa' 'Iris-virginica' 'Iris-versicolor' 'Iris-versicolor'
 'Iris-setosa' 'Iris-versicolor' 'Iris-setosa' 'Iris-setosa'
 'Iris-virginica' 'Iris-versicolor' 'Iris-virginica' 'Iris-virginica'
 'Iris-virginica' 'Iris-versicolor' 'Iris-setosa' 'Iris-setosa'
 'Iris-setosa' 'Iris-versicolor' 'Iris-versicolor' 'Iris-virginica'
 'Iris-setosa' 'Iris-virginica' 'Iris-versicolor' 'Iris-virginica'
 'Iris-virginica' 'Iris-virginica' 'Iris-versicolor' 'Iris-setosa'
 'Iris-virginica' 'Iris-setosa']
<ArrowStringArray>
[    'Iris-setosa',  'Iris-virginica', 'Iris-versicolor', 'Iris-versicolor',
     'Iris-setosa', 'Iris-versicolor',     'Iris-setosa',     'Iris-setosa',
  'Iris-virginica', 'Iris-versicolor',  'Iris-virginica',  'Iris-virginica',
  'Iris-virginica', 'Iris-versicolor',     'Iris-setosa',     'Iris-setosa',
     'Iris-setosa', 'Iris-versicolor', 'Iris-versicolor',  'Iris-virginica',
     'Iris-setosa',  'Iris-virginica', 'Iris-versicolor',  'Iris-virginica',
  'Iris-virginica', '

In [67]:
ac_score = metrics.accuracy_score(y_test, predict)
cl_report =  metrics.classification_report(y_test, predict)
print("정답룰 =", ac_score)
print("리포트 =\n", cl_report)

정답룰 = 0.9666666666666667
리포트 =
                  precision    recall  f1-score   support

    Iris-setosa       1.00      1.00      1.00        10
Iris-versicolor       1.00      0.90      0.95        10
 Iris-virginica       0.91      1.00      0.95        10

       accuracy                           0.97        30
      macro avg       0.97      0.97      0.97        30
   weighted avg       0.97      0.97      0.97        30



# ⑨ 예측 동작 확인


In [68]:
print(clf.predict(scaler.transform(pd.DataFrame(
    [[5.4, 3.3, 1.4, 0.2]],
    columns=['SepalLength', 'SepalWidth', 'PetalLength', 'PetalWidth'],
))))

['Iris-setosa']


# ⑩ 모델 & 스케일러 저장하기
   모델 -> 파일명 iris_model.pkl
   스케일러 -> 파일명 iris_scaler.pkl

   (위 저장 정보는 웹 애플리케이션에서 사용될것임)

In [69]:
model_path = os.path.join('.', r'iris_model.pkl')
joblib.dump(clf, model_path)

['./iris_model.pkl']

In [70]:
model_path = os.path.join('.', r'iris_scaler.pkl')
joblib.dump(scaler, model_path)

['./iris_scaler.pkl']

# ⑪ 저장된 모델 & 스케일러 불러오기

In [71]:
model_path = os.path.join('.', r'iris_model.pkl')
clf = joblib.load(model_path)
clf

,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",0.1
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo random number generation for shuffling the data forprobability estimates. Ignored when `probability` is False.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide <scores_probabilities>`...deprecated:: 1.9 The `probability` parameter is deprecated and will be removed in 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`.",'deprecated'
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None


In [72]:
model_path = os.path.join('.', r'iris_scaler.pkl')
scaler = joblib.load(model_path)
scaler

StandardScaler()

# ⑫ 예측 함수 만들어 보고 동작 시키기

In [73]:
def predict_iris(sepal_length, sepal_width, petal_length, petal_width):
    iris = pd.DataFrame(
        [[sepal_length, sepal_width, petal_length, petal_width]],
        columns=['SepalLength', 'SepalWidth', 'PetalLength', 'PetalWidth'],
    )

    X_scaled = scaler.transform(iris)

    result = clf.predict(X_scaled)

    return result[0]

In [74]:
print(predict_iris(5.4, 3.3, 1.4, 0.2))

Iris-setosa
